                 PHASE 1
                   
Movie metadata

      ↓

combined text

      ↓

SBERT

      ↓

item embeddings

      ↓

movie embedding space


                 PHASE 2

User's TRAIN history

      ↓

movies watched

      ↓

their item embeddings

      ↓

engagement weights

      ↓

weighted average

      ↓

USER EMBEDDING

      ↓

cosine similarity

      ↓
      
candidate movies


# Phase 2 — Personalized User Embeddings and Candidate Retrieval

### Objective

Phase 1 generated:
- cleaned movie/item catalog
- movie watch history
- item embeddings using SBERT

Phase 2 will:

1. Load the Phase-1 outputs.
2. Build a mapping from movie ID → item embedding.
3. Use only training interactions to construct user embeddings.
4. Weight movies using watch time and recency.
5. Retrieve candidate movies using cosine similarity / FAISS.
6. Remove movies already watched by the user.
7. Evaluate retrieval against the held-out test interactions.

### Important

The test interactions are never used while constructing user embeddings.
This prevents data leakage.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import faiss

from sklearn.preprocessing import normalize
from sentence_transformers.util import cos_sim

c:\Users\nihal\torch_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_PATH = r"C:\nihal\rough\Reccomender_system_rough"

DATA_PATH = os.path.join(
    BASE_PATH,
    "Cleaned_dataset"
)

print(DATA_PATH)

C:\nihal\rough\Reccomender_system_rough\Cleaned_dataset


In [3]:
items = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "items.csv"
    )
)

print("Items shape:", items.shape)
print(items.columns.tolist())

Items shape: (8787, 12)
['id', 'source', 'title', 'domain', 'metadata', 'description', 'release_year', 'duration', 'rating', 'language', 'country', 'combined']


In [4]:
movie_items = items[
    items['source'].astype(str).str.lower() == 'movies'
].copy()

movie_items = movie_items.reset_index(drop=True)

print("Movie items:", len(movie_items))
print(movie_items.head())

Movie items: 1000
           id  source            title           domain          metadata  \
0  movie_0001  movies    dragon legend  stand-up comedy  history thriller   
1  movie_0002  movies    storm warrior  stand-up comedy            sci-fi   
2  movie_0003  movies      fire family            movie             drama   
3  movie_0004  movies     our princess      documentary            sci-fi   
4  movie_0005  movies  warrior mission      documentary     sport mystery   

  description  release_year duration rating  language country  \
0         NaN          2014     35.0   TV-Y    french   japan   
1         NaN          2017     37.0     PG  japanese     usa   
2         NaN          2003    142.0  TV-MA   english     usa   
3         NaN          2011    131.0  NC-17  japanese     usa   
4         NaN          2015     91.0   TV-G   english     usa   

                                            combined  
0  Title: dragon legend Domain: stand-up comedy M...  
1  Title: storm wa

In [5]:
print(items['source'].value_counts(dropna=False))

source
netflix    7787
movies     1000
Name: count, dtype: int64


In [6]:
item_embeddings = torch.load(
    os.path.join(
        DATA_PATH,
        "item_embeddings.pt"
    ),
    map_location="cpu"
)

print(type(item_embeddings))
print(item_embeddings.shape)

<class 'torch.Tensor'>
torch.Size([8787, 384])


C:\Users\nihal\AppData\Local\Temp\ipykernel_42944\2195708783.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  item_embeddings = torch.load(


In [7]:
assert len(items) == item_embeddings.shape[0]

assert movie_items['id'].is_unique

print("Movie item ↔ embedding alignment verified.")

Movie item ↔ embedding alignment verified.


In [8]:
movie_indices = items[
    items['source'].astype(str).str.lower() == 'movies'
].index

movie_embeddings = item_embeddings[
    movie_indices
]

movie_embeddings = movie_embeddings.float()

print("Movie embeddings:", movie_embeddings.shape)

Movie embeddings: torch.Size([1000, 384])


In [9]:
movie_embeddings = torch.nn.functional.normalize(
    movie_embeddings,
    p=2,
    dim=1
)

print(movie_embeddings.shape)

torch.Size([1000, 384])


In [10]:
movie_id_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(
        movie_items['id']
    )
}

print("Movie ID mappings:", len(movie_id_to_idx))

Movie ID mappings: 1000


In [11]:
assert len(movie_items) == movie_embeddings.shape[0]

assert len(movie_id_to_idx) == len(movie_items)

assert movie_items['id'].is_unique

print("Movie embedding mapping verified.")

Movie embedding mapping verified.


In [12]:
movie_history = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "movie_history.csv"
    )
)

print("Movie history:", movie_history.shape)
print(movie_history.columns.tolist())

Movie history: (105000, 22)
['session_id', 'user_id', 'item_id', 'interaction_date', 'device_type', 'time', 'progress_percentage', 'action', 'quality', 'location_country', 'is_download', 'user_rating', 'time_capped', 'time_norm', 'progress_norm', 'action_score', 'rating_score', 'engagement_score', 'title', 'combined', 'metadata', 'domain']


In [13]:
train_interactions = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "movie_train_interactions.csv"
    )
)

print("Train:", train_interactions.shape)
print(train_interactions.head())

Train: (89470, 4)
      user_id     item_id interaction_date  engagement_score
0  user_00001  movie_0693       2024-02-05          0.207000
1  user_00001  movie_0672       2024-05-06          0.391500
2  user_00001  movie_0125       2024-05-22          0.678402
3  user_00001  movie_0450       2024-06-22          0.511926
4  user_00001  movie_0665       2024-07-22          0.543260


In [14]:
test_interactions = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "movie_test_interactions.csv"
    )
)

print("Test:", test_interactions.shape)
print(test_interactions.head())

Test: (10000, 4)
      user_id     item_id interaction_date  engagement_score
0  user_00001  movie_0483       2025-10-30          0.585000
1  user_00002  movie_0362       2025-11-28          0.240696
2  user_00003  movie_0855       2025-06-24          0.321370
3  user_00004  movie_0712       2025-12-29          0.581564
4  user_00005  movie_0423       2025-09-19          0.367155


In [15]:
train_users = set(
    train_interactions['user_id']
)

test_users = set(
    test_interactions['user_id']
)

print("Train users:", len(train_users))
print("Test users:", len(test_users))

print(
    "Users present in both:",
    len(train_users & test_users)
)

Train users: 10000
Test users: 10000
Users present in both: 10000


In [16]:
valid_movie_ids = set(
    movie_items['id']
)

invalid_train = (
    ~train_interactions['item_id'].isin(
        valid_movie_ids
    )
).sum()

invalid_test = (
    ~test_interactions['item_id'].isin(
        valid_movie_ids
    )
).sum()

print("Invalid train movie IDs:", invalid_train)
print("Invalid test movie IDs:", invalid_test)

Invalid train movie IDs: 0
Invalid test movie IDs: 0


In [17]:
train_interactions['interaction_date'] = pd.to_datetime(
    train_interactions['interaction_date'],
    errors='coerce'
)

test_interactions['interaction_date'] = pd.to_datetime(
    test_interactions['interaction_date'],
    errors='coerce'
)

In [18]:
last_train = (
    train_interactions
    .groupby('user_id')['interaction_date']
    .max()
    .reset_index(name='last_train_date')
)

test_dates = (
    test_interactions[
        ['user_id', 'interaction_date']
    ]
)

temporal_check = last_train.merge(
    test_dates,
    on='user_id',
    how='inner'
)

print(
    "Temporal split valid:",
    (
        temporal_check['last_train_date']
        <
        temporal_check['interaction_date']
    ).all()
)

Temporal split valid: False


In [19]:
train_interactions.head()

,user_id,item_id,interaction_date,engagement_score
0,user_00001,movie_0693,2024-02-05,0.207000
1,user_00001,movie_0672,2024-05-06,0.391500
2,user_00001,movie_0125,2024-05-22,0.678402
3,user_00001,movie_0450,2024-06-22,0.511926
4,user_00001,movie_0665,2024-07-22,0.543260


In [20]:
train_interactions['interaction_date'] = pd.to_datetime(
    train_interactions['interaction_date'],
    errors='coerce'
)

test_interactions['interaction_date'] = pd.to_datetime(
    test_interactions['interaction_date'],
    errors='coerce'
)

In [21]:
user_latest_date = (
    train_interactions
    .groupby('user_id')['interaction_date']
    .transform('max')
)

In [22]:
train_interactions['month_diff'] = (
    (user_latest_date.dt.year -
     train_interactions['interaction_date'].dt.year) * 12
    +
    (user_latest_date.dt.month -
     train_interactions['interaction_date'].dt.month)
)

In [23]:
train_interactions.head(10)

,user_id,item_id,interaction_date,engagement_score,month_diff
0,user_00001,movie_0693,2024-02-05,0.207000,17
1,user_00001,movie_0672,2024-05-06,0.391500,14
2,user_00001,movie_0125,2024-05-22,0.678402,14
3,user_00001,movie_0450,2024-06-22,0.511926,13
4,user_00001,movie_0665,2024-07-22,0.543260,12
5,user_00001,movie_0590,2024-11-19,0.320500,8
6,user_00001,movie_0774,2024-12-13,0.322981,7
7,user_00001,movie_0791,2025-01-26,0.602509,6
8,user_00001,movie_0225,2025-03-14,0.784617,4
9,user_00001,movie_0358,2025-05-01,0.611589,2


In [24]:
train_interactions['weight'] = (
    train_interactions['engagement_score']
    *
    np.exp(
        -0.3 * train_interactions['month_diff']
    )
)

In [25]:
train_interactions['weight'].describe()

count    89470.000000
mean         0.112530
std          0.163606
min          0.000070
25%          0.005626
50%          0.030877
75%          0.154455
max          0.991500
Name: weight, dtype: float64

In [26]:
train_interactions['weight'] = (
    train_interactions['weight']
    .fillna(0)
)

train_interactions['weight'] = (
    train_interactions['weight']
    .clip(lower=0)
)

In [27]:
def build_user_embedding(
    user_history,
    movie_id_to_idx,
    movie_embeddings
):

    embeddings = []
    weights = []

    for _, row in user_history.iterrows():

        movie_id = row['item_id']

        if movie_id not in movie_id_to_idx:
            continue

        idx = movie_id_to_idx[movie_id]

        embeddings.append(
            movie_embeddings[idx]
        )

        weights.append(
            float(row['weight'])
        )

    if len(embeddings) == 0:
        return None

    embeddings = torch.stack(
        embeddings
    )

    weights = torch.tensor(
        weights,
        dtype=embeddings.dtype
    ).reshape(-1, 1)

    # If all weights are zero,
    # use equal weighting
    if weights.sum() <= 0:

        weights = torch.ones_like(
            weights
        )

    user_embedding = (
        (embeddings * weights).sum(
            dim=0
        )
        /
        weights.sum()
    )

    # Normalize user vector
    user_embedding = torch.nn.functional.normalize(
        user_embedding.unsqueeze(0),
        p=2,
        dim=1
    ).squeeze(0)

    return user_embedding

In [28]:
user_id = train_interactions[
    'user_id'
].iloc[0]

user_history = train_interactions[
    train_interactions['user_id'] == user_id
]

user_vec = build_user_embedding(
    user_history,
    movie_id_to_idx,
    movie_embeddings
)

print("User:", user_id)
print("History size:", len(user_history))
print("Embedding shape:", user_vec.shape)

User: user_00001
History size: 11
Embedding shape: torch.Size([384])


In [29]:
user_embeddings = {}

for user_id, group in train_interactions.groupby(
    'user_id'
):

    user_vec = build_user_embedding(
        group,
        movie_id_to_idx,
        movie_embeddings
    )

    if user_vec is not None:
        user_embeddings[user_id] = user_vec

In [30]:
print(
    "Number of user embeddings:",
    len(user_embeddings)
)

print(
    "Expected train users:",
    train_interactions['user_id'].nunique()
)

Number of user embeddings: 10000
Expected train users: 10000


In [31]:
first_user = next(
    iter(user_embeddings)
)

print(
    user_embeddings[first_user].shape
)

torch.Size([384])


In [32]:
PHASE2_PATH = os.path.join(
    BASE_PATH,
    "Phase_2"
)

os.makedirs(
    PHASE2_PATH,
    exist_ok=True
)

In [33]:
torch.save(
    user_embeddings,
    os.path.join(
        PHASE2_PATH,
        "user_embeddings.pt"
    )
)

print(
    "User embeddings saved."
)

User embeddings saved.


In [34]:
movie_matrix = (
    movie_embeddings
    .cpu()
    .numpy()
    .astype('float32')
)

index = faiss.IndexFlatIP(
    movie_matrix.shape[1]
)

index.add(movie_matrix)

print(
    "FAISS index size:",
    index.ntotal
)

FAISS index size: 1000


In [35]:
movie_matrix = (
    movie_embeddings
    .cpu()
    .numpy()
    .astype('float32')
)

index = faiss.IndexFlatIP(
    movie_matrix.shape[1]
)

index.add(movie_matrix)

print(
    "FAISS index size:",
    index.ntotal
)

FAISS index size: 1000


In [36]:
def recommend(
    user_id,
    train_interactions,
    movie_items,
    user_embeddings,
    index,
    k=10
):

    if user_id not in user_embeddings:
        return pd.DataFrame()

    user_vec = user_embeddings[user_id]

    query = (
        user_vec
        .cpu()
        .numpy()
        .astype('float32')
        .reshape(1, -1)
    )

    # Retrieve extra candidates because
    # some may already be watched
    search_k = min(
        k + 50,
        index.ntotal
    )

    scores, indices = index.search(
        query,
        search_k
    )

    watched = set(
        train_interactions[
            train_interactions['user_id'] == user_id
        ]['item_id']
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        movie_id = movie_items.iloc[idx]['id']

        if movie_id in watched:
            continue

        row = movie_items.iloc[idx]

        results.append({
            'movie_id': movie_id,
            'title': row['title'],
            'domain': row['domain'],
            'metadata': row['metadata'],
            'score': float(score)
        })

        if len(results) == k:
            break

    return pd.DataFrame(results)

In [37]:
user_id = train_interactions[
    'user_id'
].iloc[0]

recommendations = recommend(
    user_id,
    train_interactions,
    movie_items,
    user_embeddings,
    index,
    k=10
)

recommendations

,movie_id,title,domain,metadata,score
0,movie_0159,battle family,movie,drama mystery,0.910255
1,movie_0361,was adventure,movie,drama,0.904871
2,movie_0840,family kingdom,movie,adventure,0.899597
3,movie_0592,king adventure,movie,drama,0.899525
4,movie_0189,family mission,movie,documentary,0.891826
5,movie_0003,fire family,movie,drama,0.886919
6,movie_0916,adventure dream,movie,documentary,0.885716
7,movie_0611,big adventure,movie,documentary fantasy,0.884585
8,movie_0431,a adventure,movie,history,0.883777
9,movie_0789,adventure quest,movie,action drama,0.883368


In [38]:
user_id = train_interactions[
    'user_id'
].iloc[0]

recommendations = recommend(
    user_id,
    train_interactions,
    movie_items,
    user_embeddings,
    index,
    k=10
)

recommendations

,movie_id,title,domain,metadata,score
0,movie_0159,battle family,movie,drama mystery,0.910255
1,movie_0361,was adventure,movie,drama,0.904871
2,movie_0840,family kingdom,movie,adventure,0.899597
3,movie_0592,king adventure,movie,drama,0.899525
4,movie_0189,family mission,movie,documentary,0.891826
5,movie_0003,fire family,movie,drama,0.886919
6,movie_0916,adventure dream,movie,documentary,0.885716
7,movie_0611,big adventure,movie,documentary fantasy,0.884585
8,movie_0431,a adventure,movie,history,0.883777
9,movie_0789,adventure quest,movie,action drama,0.883368


In [39]:
watched = set(
    train_interactions[
        train_interactions['user_id'] == user_id
    ]['item_id']
)

recommended = set(
    recommendations['movie_id']
)

print(
    "Watched recommendations:",
    len(watched & recommended)
)

Watched recommendations: 0


In [40]:
recommendations[
    [
        'title',
        'domain',
        'metadata',
        'score'
    ]
]

,title,domain,metadata,score
0,battle family,movie,drama mystery,0.910255
1,was adventure,movie,drama,0.904871
2,family kingdom,movie,adventure,0.899597
3,king adventure,movie,drama,0.899525
4,family mission,movie,documentary,0.891826
5,fire family,movie,drama,0.886919
6,adventure dream,movie,documentary,0.885716
7,big adventure,movie,documentary fantasy,0.884585
8,a adventure,movie,history,0.883777
9,adventure quest,movie,action drama,0.883368


In [41]:
def get_item_embedding(
    movie_id,
    movie_id_to_idx,
    movie_embeddings
):
    """
    Return normalized embedding for a movie.
    """
    idx = movie_id_to_idx[movie_id]
    return movie_embeddings[idx]

In [42]:
def item_similarity(
    actual_movie_id,
    predicted_movie_id,
    movie_id_to_idx,
    movie_embeddings
):

    actual_emb = get_item_embedding(
        actual_movie_id,
        movie_id_to_idx,
        movie_embeddings
    )

    predicted_emb = get_item_embedding(
        predicted_movie_id,
        movie_id_to_idx,
        movie_embeddings
    )

    similarity = torch.dot(
        actual_emb,
        predicted_emb
    ).item()

    return similarity

In [43]:
SEMANTIC_THRESHOLD = 0.80

In [44]:
user_id = test_interactions['user_id'].iloc[0]

actual_movie = test_interactions[
    test_interactions['user_id'] == user_id
]['item_id'].iloc[0]

predicted_movie = recommendations[
    'movie_id'
].iloc[0]

sim = item_similarity(
    actual_movie,
    predicted_movie,
    movie_id_to_idx,
    movie_embeddings
)

print("Actual:", actual_movie)
print("Predicted:", predicted_movie)
print("Cosine similarity:", sim)

Actual: movie_0483
Predicted: movie_0159
Cosine similarity: 0.8561640977859497


In [45]:
def is_semantically_relevant(
    similarity,
    threshold=SEMANTIC_THRESHOLD
):
    return similarity >= threshold

This is different from normal Recall@K.

Normal Recall:

predicted movie_id == actual movie_id

Semantic Recall:

cosine(predicted_movie, actual_movie)
    >= threshold

In [46]:
def semantic_recall_at_k(
    relevant_items,
    predicted_items,
    movie_id_to_idx,
    movie_embeddings,
    k=10,
    threshold=0.80
):

    if len(relevant_items) == 0:
        return 0.0

    predicted_items = predicted_items[:k]

    for actual_movie in relevant_items:

        if actual_movie not in movie_id_to_idx:
            continue

        for predicted_movie in predicted_items:

            if predicted_movie not in movie_id_to_idx:
                continue

            similarity = item_similarity(
                actual_movie,
                predicted_movie,
                movie_id_to_idx,
                movie_embeddings
            )

            if similarity >= threshold:
                return 1.0

    return 0.0

In [47]:
def semantic_mrr(
    relevant_items,
    predicted_items,
    movie_id_to_idx,
    movie_embeddings,
    threshold=0.80
):

    if len(relevant_items) == 0:
        return 0.0

    relevant_items = set(relevant_items)

    for rank, predicted_movie in enumerate(
        predicted_items,
        start=1
    ):

        if predicted_movie not in movie_id_to_idx:
            continue

        for actual_movie in relevant_items:

            if actual_movie not in movie_id_to_idx:
                continue

            similarity = item_similarity(
                actual_movie,
                predicted_movie,
                movie_id_to_idx,
                movie_embeddings
            )

            if similarity >= threshold:
                return 1.0 / rank

    return 0.0

In [48]:
def semantic_ndcg_at_k(
    relevant_items,
    predicted_items,
    movie_id_to_idx,
    movie_embeddings,
    k=10
):

    if len(relevant_items) == 0:
        return 0.0

    predicted_items = predicted_items[:k]

    # In your current setup there is normally
    # one held-out test item per user.
    actual_movie = relevant_items[0]

    if actual_movie not in movie_id_to_idx:
        return 0.0

    similarities = []

    for predicted_movie in predicted_items:

        if predicted_movie not in movie_id_to_idx:
            similarities.append(0.0)
            continue

        similarity = item_similarity(
            actual_movie,
            predicted_movie,
            movie_id_to_idx,
            movie_embeddings
        )

        # Cosine similarity can theoretically be negative.
        # Relevance cannot be negative.
        relevance = max(similarity, 0.0)

        similarities.append(relevance)

    # DCG using the recommendation ranking
    dcg = 0.0

    for rank, relevance in enumerate(
        similarities,
        start=1
    ):

        dcg += (
            relevance /
            np.log2(rank + 1)
        )

    # Ideal ranking:
    # sort the SAME candidate relevance scores
    # from highest to lowest.
    ideal_similarities = sorted(
        similarities,
        reverse=True
    )

    idcg = 0.0

    for rank, relevance in enumerate(
        ideal_similarities,
        start=1
    ):

        idcg += (
            relevance /
            np.log2(rank + 1)
        )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [49]:
def precision_at_k(
    relevant,
    predicted,
    k
):

    predicted = predicted[:k]

    if k == 0:
        return 0.0

    hits = len(
        set(predicted) &
        set(relevant)
    )

    return hits / k

In [50]:
def average_similarity_at_k(
    actual_movie,
    predicted_items,
    movie_id_to_idx,
    movie_embeddings,
    k=10
):

    if actual_movie not in movie_id_to_idx:
        return 0.0

    similarities = []

    for predicted_movie in predicted_items[:k]:

        if predicted_movie not in movie_id_to_idx:
            continue

        similarity = item_similarity(
            actual_movie,
            predicted_movie,
            movie_id_to_idx,
            movie_embeddings
        )

        similarities.append(similarity)

    if not similarities:
        return 0.0

    return np.mean(similarities)

In [51]:
def max_similarity_at_k(
    actual_movie,
    predicted_items,
    movie_id_to_idx,
    movie_embeddings,
    k=10
):

    if actual_movie not in movie_id_to_idx:
        return 0.0

    similarities = []

    for predicted_movie in predicted_items[:k]:

        if predicted_movie not in movie_id_to_idx:
            continue

        similarity = item_similarity(
            actual_movie,
            predicted_movie,
            movie_id_to_idx,
            movie_embeddings
        )

        similarities.append(similarity)

    if not similarities:
        return 0.0

    return max(similarities)

In [52]:
def recall_at_k(
    relevant,
    predicted,
    k
):

    if len(relevant) == 0:
        return 0.0

    predicted = predicted[:k]

    hits = len(
        set(predicted) &
        set(relevant)
    )

    return hits / len(relevant)

In [53]:
def reciprocal_rank(
    relevant,
    predicted
):

    relevant = set(relevant)

    for rank, item_id in enumerate(
        predicted,
        start=1
    ):

        if item_id in relevant:
            return 1.0 / rank

    return 0.0

In [54]:
def ndcg_at_k(
    relevant,
    predicted,
    k
):

    relevant = set(relevant)

    predicted = predicted[:k]

    dcg = 0.0

    for rank, item_id in enumerate(
        predicted,
        start=1
    ):

        if item_id in relevant:
            dcg += 1.0 / np.log2(
                rank + 1
            )

    ideal_hits = min(
        len(relevant),
        k
    )

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    return dcg / idcg

In [55]:
test_items = test_interactions[
    test_interactions['user_id'] == user_id
]['item_id'].tolist()

predicted_items = recommendations[
    'movie_id'
].tolist()

print(
    "Relevant:",
    test_items
)

print(
    "Predicted:",
    predicted_items
)

print(
    "Precision@10:",
    precision_at_k(
        test_items,
        predicted_items,
        10
    )
)

print(
    "Recall@10:",
    recall_at_k(
        test_items,
        predicted_items,
        10
    )
)

print(
    "MRR:",
    reciprocal_rank(
        test_items,
        predicted_items
    )
)

print(
    "NDCG@10:",
    ndcg_at_k(
        test_items,
        predicted_items,
        10
    )
)

Relevant: ['movie_0483']
Predicted: ['movie_0159', 'movie_0361', 'movie_0840', 'movie_0592', 'movie_0189', 'movie_0003', 'movie_0916', 'movie_0611', 'movie_0431', 'movie_0789']
Precision@10: 0.0
Recall@10: 0.0
MRR: 0.0
NDCG@10: 0.0


In [56]:


predictions = recommend(
    user_id,
    train_interactions,
    movie_items,
    user_embeddings,
    index,
    k=10
)

predicted = predictions[
    'movie_id'
].tolist()

relevant = test_interactions[
    test_interactions['user_id'] == user_id
]['item_id'].tolist()

print("Actual:", relevant)
print("Predicted:", predicted)

print(
    "Exact Recall@10:",
    recall_at_k(
        relevant,
        predicted,
        10
    )
)

print(
    "Exact MRR:",
    reciprocal_rank(
        relevant,
        predicted
    )
)

print(
    "Semantic Recall@10:",
    semantic_recall_at_k(
        relevant,
        predicted,
        movie_id_to_idx,
        movie_embeddings,
        k=10,
        threshold=0.80
    )
)

print(
    "Semantic MRR:",
    semantic_mrr(
        relevant,
        predicted,
        movie_id_to_idx,
        movie_embeddings,
        threshold=0.80
    )
)

print(
    "Semantic NDCG@10:",
    semantic_ndcg_at_k(
        relevant,
        predicted,
        movie_id_to_idx,
        movie_embeddings,
        k=10
    )
)


Actual: ['movie_0483']
Predicted: ['movie_0159', 'movie_0361', 'movie_0840', 'movie_0592', 'movie_0189', 'movie_0003', 'movie_0916', 'movie_0611', 'movie_0431', 'movie_0789']
Exact Recall@10: 0.0
Exact MRR: 0.0
Semantic Recall@10: 1.0
Semantic MRR: 1.0
Semantic NDCG@10: 0.9970756027392653


In [57]:
actual_movie = 'movie_0417'

avg_sim = average_similarity_at_k(
    actual_movie,
    predicted,
    movie_id_to_idx,
    movie_embeddings,
    k=10
)

max_sim = max_similarity_at_k(
    actual_movie,
    predicted,
    movie_id_to_idx,
    movie_embeddings,
    k=10
)

print("Average Similarity@10:", avg_sim)
print("Max Similarity@10:", max_sim)

Average Similarity@10: 0.6506669461727143
Max Similarity@10: 0.7332113981246948


In [58]:
actual_movie = 'movie_0417'

for rank, predicted_movie in enumerate(
    predicted,
    start=1
):

    similarity = item_similarity(
        actual_movie,
        predicted_movie,
        movie_id_to_idx,
        movie_embeddings
    )

    print(
        f"Rank {rank}: "
        f"{predicted_movie} "
        f"→ similarity = {similarity:.4f}"
    )

Rank 1: movie_0159 → similarity = 0.6022
Rank 2: movie_0361 → similarity = 0.6248
Rank 3: movie_0840 → similarity = 0.6329
Rank 4: movie_0592 → similarity = 0.6586
Rank 5: movie_0189 → similarity = 0.6754
Rank 6: movie_0003 → similarity = 0.5815
Rank 7: movie_0916 → similarity = 0.7243
Rank 8: movie_0611 → similarity = 0.7332
Rank 9: movie_0431 → similarity = 0.6579
Rank 10: movie_0789 → similarity = 0.6158


In [59]:
for rank, movie_id in enumerate(
    predicted,
    start=1
):

    row = movie_items[
        movie_items['id'] == movie_id
    ].iloc[0]

    similarity = item_similarity(
        'movie_0417',
        movie_id,
        movie_id_to_idx,
        movie_embeddings
    )

    print(
        f"{rank}. {row['title']} "
        f"| {row['domain']} "
        f"| {similarity:.4f}"
    )

1. battle family | movie | 0.6022
2. was adventure | movie | 0.6248
3. family kingdom | movie | 0.6329
4. king adventure | movie | 0.6586
5. family mission | movie | 0.6754
6. fire family | movie | 0.5815
7. adventure dream | movie | 0.7243
8. big adventure | movie | 0.7332
9. a adventure | movie | 0.6579
10. adventure quest | movie | 0.6158


In [60]:
actual_movie = 'movie_0483'

for rank, predicted_movie in enumerate(
    predicted,
    start=1
):

    similarity = item_similarity(
        actual_movie,
        predicted_movie,
        movie_id_to_idx,
        movie_embeddings
    )

    print(
        f"Rank {rank}: "
        f"{predicted_movie} "
        f"→ similarity = {similarity:.4f}"
    )

Rank 1: movie_0159 → similarity = 0.8562
Rank 2: movie_0361 → similarity = 0.8024
Rank 3: movie_0840 → similarity = 0.8165
Rank 4: movie_0592 → similarity = 0.8009
Rank 5: movie_0189 → similarity = 0.8413
Rank 6: movie_0003 → similarity = 0.7895
Rank 7: movie_0916 → similarity = 0.8061
Rank 8: movie_0611 → similarity = 0.7921
Rank 9: movie_0431 → similarity = 0.8034
Rank 10: movie_0789 → similarity = 0.7939


In [61]:
evaluation_results = []

test_users = test_interactions[
    'user_id'
].unique()

for user_id in test_users:

    if user_id not in user_embeddings:
        continue

    predictions = recommend(
        user_id,
        train_interactions,
        movie_items,
        user_embeddings,
        index,
        k=10
    )

    predicted = predictions[
        'movie_id'
    ].tolist()

    relevant = test_interactions[
        test_interactions['user_id'] == user_id
    ]['item_id'].tolist()

    if len(relevant) == 0:
        continue

    evaluation_results.append({

        'user_id': user_id,

        # Exact metrics
        'Precision@10':
            precision_at_k(
                relevant,
                predicted,
                10
            ),

        'Recall@10':
            recall_at_k(
                relevant,
                predicted,
                10
            ),

        'MRR':
            reciprocal_rank(
                relevant,
                predicted
            ),

        'NDCG@10':
            ndcg_at_k(
                relevant,
                predicted,
                10
            ),

        # Semantic metrics
        'SemanticRecall@10':
            semantic_recall_at_k(
                relevant,
                predicted,
                movie_id_to_idx,
                movie_embeddings,
                k=10,
                threshold=0.80
            ),

        'SemanticMRR':
            semantic_mrr(
                relevant,
                predicted,
                movie_id_to_idx,
                movie_embeddings,
                threshold=0.80
            ),

        'SemanticNDCG@10':
            semantic_ndcg_at_k(
                relevant,
                predicted,
                movie_id_to_idx,
                movie_embeddings,
                k=10
            )
    })

In [62]:
metrics_df = pd.DataFrame(
    evaluation_results
)

metrics_df.head()

,user_id,Precision@10,Recall@10,MRR,NDCG@10,SemanticRecall@10,SemanticMRR,SemanticNDCG@10
0,user_00001,0.0,0.0,0.0,0.0,1.0,1.000000,0.997076
1,user_00002,0.0,0.0,0.0,0.0,1.0,0.111111,0.977550
2,user_00003,0.0,0.0,0.0,0.0,1.0,0.250000,0.985100
3,user_00004,0.0,0.0,0.0,0.0,1.0,0.500000,0.965539
4,user_00005,0.0,0.0,0.0,0.0,0.0,0.000000,0.954697


In [63]:
phase2_metrics = metrics_df[
    [
         'Precision@10',
        'Recall@10',
        'MRR',
        'NDCG@10',
        'SemanticRecall@10',
        'SemanticMRR',
        'SemanticNDCG@10'
    ]
].mean()

phase2_metrics

Precision@10         0.001110
Recall@10            0.011100
MRR                  0.003317
NDCG@10              0.005111
SemanticRecall@10    0.290900
SemanticMRR          0.138038
SemanticNDCG@10      0.975796
dtype: float64

In [64]:
print(
    "Evaluated users:",
    len(metrics_df)
)

print(
    "Total test users:",
    len(test_users)
)

Evaluated users: 10000
Total test users: 10000


In [65]:
recommendation_rows = []

for user_id in test_users:

    if user_id not in user_embeddings:
        continue

    predictions = recommend(
        user_id,
        train_interactions,
        movie_items,
        user_embeddings,
        index,
        k=10
    )

    for rank, (_, row) in enumerate(
        predictions.iterrows(),
        start=1
    ):

        recommendation_rows.append({
            'user_id': user_id,
            'rank': rank,
            'movie_id': row['movie_id'],
            'title': row['title'],
            'score': row['score']
        })

In [66]:
recommendations_df = pd.DataFrame(
    recommendation_rows
)

recommendations_df.to_csv(
    os.path.join(
        PHASE2_PATH,
        "phase2_recommendations.csv"
    ),
    index=False
)

print(
    "Recommendations saved."
)

Recommendations saved.


In [67]:
phase2_metrics_df = pd.DataFrame(
    [
        {
            'model': 'Phase 2 - Weighted User Embedding',
            'Precision@10': phase2_metrics[
                'Precision@10'
            ],
            'Recall@10': phase2_metrics[
                'Recall@10'
            ],
            'MRR': phase2_metrics[
                'MRR'
            ],
            'NDCG@10': phase2_metrics[
                'NDCG@10'
            ],
             'SemanticRecall@10':phase2_metrics['SemanticRecall@10'],
                    'SemanticMRR':phase2_metrics['SemanticMRR'],
                    'SemanticNDCG@10':phase2_metrics['SemanticNDCG@10']
        }
    ]
)

phase2_metrics_df

,model,Precision@10,Recall@10,MRR,NDCG@10,SemanticRecall@10,SemanticMRR,SemanticNDCG@10
0,Phase 2 - Weighted User Embedding,0.00111,0.0111,0.003317,0.005111,0.2909,0.138038,0.975796


In [68]:
def evaluate_semantic_threshold(
    test_interactions,
    train_interactions,
    user_embeddings,
    movie_items,
    index,
    movie_id_to_idx,
    movie_embeddings,
    threshold,
    k=10
):

    recalls = []
    mrrs = []

    for user_id in test_interactions['user_id'].unique():

        if user_id not in user_embeddings:
            continue

        predictions = recommend(
            user_id,
            train_interactions,
            movie_items,
            user_embeddings,
            index,
            k=k
        )

        predicted = predictions[
            'movie_id'
        ].tolist()

        relevant = test_interactions[
            test_interactions['user_id'] == user_id
        ]['item_id'].tolist()

        if len(relevant) == 0:
            continue

        recalls.append(
            semantic_recall_at_k(
                relevant,
                predicted,
                movie_id_to_idx,
                movie_embeddings,
                k=k,
                threshold=threshold
            )
        )

        mrrs.append(
            semantic_mrr(
                relevant,
                predicted,
                movie_id_to_idx,
                movie_embeddings,
                threshold=threshold
            )
        )

    return {
        'threshold': threshold,
        'SemanticRecall@10': np.mean(recalls),
        'SemanticMRR': np.mean(mrrs)
    }

In [69]:
threshold_results = []

for threshold in [
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95
]:

    result = evaluate_semantic_threshold(
        test_interactions,
        train_interactions,
        user_embeddings,
        movie_items,
        index,
        movie_id_to_idx,
        movie_embeddings,
        threshold,
        k=10
    )

    threshold_results.append(result)

In [70]:
threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

,threshold,SemanticRecall@10,SemanticMRR
0,0.70,0.7246,0.493837
1,0.75,0.5112,0.300526
2,0.80,0.2909,0.138038
3,0.85,0.1164,0.044969
4,0.90,0.0287,0.009049
5,0.95,0.0130,0.004034


In [71]:
actual_row = movie_items[
    movie_items['id'] == 'movie_0483'
]

actual_row[
    ['id', 'title', 'domain', 'metadata']
]

,id,title,domain,metadata
482,movie_0483,war journey,movie,family thriller


In [72]:
predicted_rows = movie_items[
    movie_items['id'].isin(predicted)
][
    ['id', 'title', 'domain', 'metadata']
]

predicted_rows

,id,title,domain,metadata
155,movie_0156,an ice,movie,history thriller
228,movie_0229,last secret,movie,drama
281,movie_0282,secret ice,movie,comedy drama
436,movie_0437,ice secret,movie,animation drama
480,movie_0481,ice story,movie,war
582,movie_0583,hero ice,movie,comedy
822,movie_0823,adventure ice,movie,comedy
916,movie_0917,adventure ice,movie,adventure
953,movie_0954,mission adventure,movie,comedy
998,movie_0999,ice war,movie,crime


Exact Recall@10       = 0

Exact MRR             = 0

Semantic Recall@10    = 1   ← valid for this user

Semantic MRR          = 1   ← valid for this user

Semantic NDCG@10      ≤ 1   ← after fixing the implementation

In [73]:
for rank, movie_id in enumerate(
    predicted,
    start=1
):

    similarity = item_similarity(
        'movie_0483',
        movie_id,
        movie_id_to_idx,
        movie_embeddings
    )

    title = movie_items.loc[
        movie_items['id'] == movie_id,
        'title'
    ].iloc[0]

    print(
        f"{rank}. {title} "
        f"→ {similarity:.4f}"
    )

1. secret ice → 0.6774
2. adventure ice → 0.7015
3. ice war → 0.7439
4. adventure ice → 0.7473
5. ice secret → 0.6678
6. an ice → 0.7711
7. ice story → 0.7915
8. mission adventure → 0.7845
9. last secret → 0.7795
10. hero ice → 0.6717


In [ ]:
test_actuals = (
    test_interactions[
        ['user_id', 'item_id']
    ]
    .rename(
        columns={
            'item_id': 'actual_movie_id'
        }
    )
)

In [ ]:
prediction_rows = []

for user_id in test_interactions['user_id'].unique():

    if user_id not in user_embeddings:
        continue

    recommendations = recommend(
        user_id,
        train_interactions,
        movie_items,
        user_embeddings,
        index,
        k=10
    )

    for rank, (_, row) in enumerate(
        recommendations.iterrows(),
        start=1
    ):

        prediction_rows.append({
            'user_id': user_id,
            'rank': rank,
            'movie_id': row['movie_id'],
            'title': row['title'],
            'domain': row['domain'],
            'score': row['score']
        })

In [ ]:
predictions_df = pd.DataFrame(
    prediction_rows
)

print(
    "Prediction shape:",
    predictions_df.shape
)

predictions_df.head(20)

In [ ]:
predictions_df = predictions_df.merge(
    test_actuals,
    on='user_id',
    how='left'
)

In [ ]:
predictions_df['exact_match'] = (
    predictions_df['movie_id']
    ==
    predictions_df['actual_movie_id']
)

In [ ]:
def get_similarity_to_actual(row):

    actual = row['actual_movie_id']
    predicted = row['movie_id']

    if (
        actual not in movie_id_to_idx
        or
        predicted not in movie_id_to_idx
    ):
        return np.nan

    return item_similarity(
        actual,
        predicted,
        movie_id_to_idx,
        movie_embeddings
    )

In [ ]:
predictions_df['similarity_to_actual'] = (
    predictions_df.apply(
        get_similarity_to_actual,
        axis=1
    )
)

In [ ]:
prediction_path = os.path.join(
    PHASE2_PATH,
    "phase2_predictions.csv"
)

predictions_df.to_csv(
    prediction_path,
    index=False
)

print(
    "Predictions saved at:",
    prediction_path
)

In [74]:
phase2_metrics_df.to_csv(
    os.path.join(
        PHASE2_PATH,
        "phase2_metrics.csv"
    ),
    index=False
)

print(
    "Metrics saved."
)

Metrics saved.


In [75]:
print("=" * 60)
print("PHASE 2 COMPLETE")
print("=" * 60)

print(
    "Movie items:",
    len(movie_items)
)

print(
    "Training interactions:",
    len(train_interactions)
)

print(
    "Test interactions:",
    len(test_interactions)
)

print(
    "User embeddings:",
    len(user_embeddings)
)

print(
    "FAISS items:",
    index.ntotal
)

print()

print(
    phase2_metrics_df.to_string(
        index=False
    )
)

PHASE 2 COMPLETE
Movie items: 1000
Training interactions: 89470
Test interactions: 10000
User embeddings: 10000
FAISS items: 1000

                            model  Precision@10  Recall@10      MRR  NDCG@10  SemanticRecall@10  SemanticMRR  SemanticNDCG@10
Phase 2 - Weighted User Embedding       0.00111     0.0111 0.003317 0.005111             0.2909     0.138038         0.975796


| Metric                 | What it measures                                    |
| ---------------------- | --------------------------------------------------- |
| **Precision@10**       | Exact recommended items that were actually consumed |
| **Recall@10**          | Exact test items recovered                          |
| **MRR**                | How high the first exact match appears              |
| **NDCG@10**            | Ranking quality of exact matches                    |
| **Semantic Recall@10** | Whether recommended items are semantically close    |
| **Semantic NDCG@10**   | Ranking quality using semantic relevance            |


This is actually a useful baseline. It suggests:

The weighted user embedding is capturing some of the user's content preferences, but it is not yet good at retrieving the exact future movie.

That gives us a clear target for Phase 3: improve candidate retrieval/ranking rather than changing the evaluation to make the scores look better.